# Demo Notebook for crystal_im_py

This notebook will implement the various functions in the repository crystal_im_py on a demo dataset. The goal is to walk through all the steps of uploading and analyzing microscopy image data to get thermodynamic parameters using a series of functions with customizable input so that you can run this notebook on your own data in the future.

## Step 0: Setup

If you're using this notebook, you should have already cloned the GitHub repository and have Jupyter notebook open in your browser. This setup code allows you to call the functions in crystal_tools throughout this notebook.

In [1]:
import sys
import os

# Add 'crystal_im_py' to the path so 'crystal_tools' is recognized as a package
sys.path.append(os.path.abspath(".."))

## Step 1: Upload Images and Create Dataframes

These functions upload multiple experiments of microscopy data from the same crystal conducted at varying temperatures and create a list of dataframes corresponding to the collection of experiments.

1. **load_images_from_folder:** Upload images from raw files and convert into scikit-image arrays
2. **calculate_channel_intensities:** Extract mean intensity of desired channels from scikit-image arrays and convert into pandas dataframe
3. **add_temp_data:** Interpolate temperature data from .csv file to assign a temperature to each timepoint in the dataframe

**Requirements:**
* Each experiment's images should be stored in their own folder with the same naming convention and the last 3 digits of the folder name corresponding to the experimental temperature
* The individual experiments' folders should be stored in a single enclosing folder (here, *'batch_images'*)
* Default image extension is .png but this can be changed in the function argument
* The timestamp for each image is extracted using naming conventions from Zeiss ZEN blue edition, which consist of a lowercase *t* and then a number (does not need leading 0) that is multiplied by the defined time interval below
* Temperature data for each experiment should be recorded in a .csv file with the same naming convention and the last 3 digits of the folder name corresponding to the experimental temperature
* For each temperature dataset, there should be two columns in the .csv with headings *Time (s)* and *Temperature (°C)* (temperature should be recorded in/converted to °C)
* **Important: the data files need to be in the same folder as this notebook!**

In [2]:
temps_list = ['075', '090', '105', '120', '135'] # enter temperatures using conventions of file names
folder_root = 'demo_data/batch_images/TOPA-1-' # enter filepath for batch image folders
temp_path = 'demo_data/temp_data/TOPA-1-' # enter filepath for temperature .csv data

time = 0.484 # enter time interval for each microscopy image (default for 'crystals_spectral' is 0.484 s)

# ---- Do not edit code below this line --- #

# Call the functions from crystal_tools for use in this code
from crystal_tools.image_loading import load_images_from_folder
from crystal_tools.data_merging import calculate_channel_intensities, add_temp_data

# Create an empty list to store resulting dataframes in
df_list = []

for temp in temps_list:
    # Load images from folder
    images, files = load_images_from_folder(f'{folder_root}{temp}', file_extension='.png')
    df = calculate_channel_intensities(images, channels={'red':0, 'green': 1, 'blue': 2}, time_interval=time)

    # Add temperature data
    temp_filename = f'{temp_path}{temp}.csv'
    add_temp_data(temp_filename, df, time)

    # Add experiment to df_list
    df_list.append(df)

Path exists: demo_data/batch_images/TOPA-1-075
Found 200 .png files
Loaded 200 images
 
Processed 200 images
Time range: 0.00s to 96.32s

DataFrame shape: (200, 4)
Columns: ['time', 'mean_red', 'mean_green', 'mean_blue']
Path exists: demo_data/batch_images/TOPA-1-090
Found 200 .png files
Loaded 200 images
 
Processed 200 images
Time range: 0.00s to 96.32s

DataFrame shape: (200, 4)
Columns: ['time', 'mean_red', 'mean_green', 'mean_blue']
Path exists: demo_data/batch_images/TOPA-1-105
Found 200 .png files
Loaded 200 images
 
Processed 200 images
Time range: 0.00s to 96.32s

DataFrame shape: (200, 4)
Columns: ['time', 'mean_red', 'mean_green', 'mean_blue']
Path exists: demo_data/batch_images/TOPA-1-120
Found 200 .png files
Loaded 200 images
 
Processed 200 images
Time range: 0.00s to 96.32s

DataFrame shape: (200, 4)
Columns: ['time', 'mean_red', 'mean_green', 'mean_blue']
Path exists: demo_data/batch_images/TOPA-1-135
Found 200 .png files
Loaded 200 images
 
Processed 200 images
Time ra

## Step 2: Reaction Analysis

This function calculates two useful parameters for further thermodynamics analysis, α (conversion ratio) and Q (reaction quotient) using the function **calculate_rxn_progress**.

**Equations:**

* $α(t) = 1 - \dfrac{I_{init}(t)}{I_{final}(t)} * \dfrac{I_{final_0}}{I_{init_0}}$

where $I_{init}(t)$ is the intensity of the initial channel as a function of time, $I_{final}(t)$ is the intensity of the final channel as a function of time, and $I_{init_0}$ and $I_{final_0}$ are the intensities of the initial and final channels at t = 0.
* $Q = \dfrac{α}{1 - α}$ using the equation for α above

**Requirements:**

In [ ]:
# Test load_init_image with demo data
filepath = 'demo_data/batch_images/TOPA-1-075/TOPA-1-075_b0v0t0z0c0-32x0-512y0-512.png'
init_0, final_0 = load_init_image(filepath, init_channel='blue', final_channel='green')